# VAE — EMNIST
**Project: Latent Diffusion on EMNIST**

This notebook trains a VAE with separate Encoder and Decoder.
At the end, it saves `encoder.ckpt` and `decoder.ckpt` for use in the Diffusion/Flow stage.

> Run on: Runtime → Change Runtime Type → T4 GPU

## 1. Imports

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import numpy as np
import os
import glob
from IPython.display import clear_output

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda


In [29]:
# Consttants
CKPT_DIR = 'training_checkpoints'  # where to save .ckpt files
RESULTS_DIR = 'results'  # where to save results

## 2. Hyperparameters

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────
LATENT_DIM        = 4           # dimensionality of the latent space z
# BATCH_SIZE        = 256
EPOCHS            = 512
EPOCHS            = 100
LEARNING_RATE     = 1e-2
USE_DIFF_SIGMA_E  = True         # True = different sigma per dimension
RECONSTRUCTION    = 'Bernoulli'  # 'Gaussian' | 'Laplacian' | 'Bernoulli'
ARCH              = '2blocks'    # '2blocks' | '3blocks'
# ────────────────────────────────────────────────────────────────

# ── Checkpointing ─────────────────────────────────────────────
GET_EARLY_TRAINED_MODEL = False  # if True, loads a pretrained model from CKPT_DIR
CHECKPOINT_EVERY = 5     
KEEP_LAST       = 5                            # save a checkpoint every N epochs
os.makedirs(CKPT_DIR, exist_ok=True)
# ────────────────────────────────────────────────────────────────

# ── Early stopping (soft / non-aggressive) ────────────────────
# Stops when EITHER of these holds, so it catches both overfitting and
# plain convergence — but is tuned to be reluctant to stop:
#   1) NO NEW MINIMUM: the test loss hasn't set a new best for
#      EARLY_STOPPING_PATIENCE epochs (any new minimum, however small,
#      resets the count).
#   2) PLATEAU: over the last EARLY_STOPPING_PATIENCE epochs the test
#      loss spread (max - min) stays below EARLY_STOPPING_MIN_DELTA.
EARLY_STOPPING            = True
EARLY_STOPPING_PATIENCE   = 40    # generous window -> waits a long time
EARLY_STOPPING_MIN_DELTA  = 0.5   # spread over the whole window (noise ~0.3)
# ────────────────────────────────────────────────────────────────


## 3. Shape helpers

In [31]:
def image_to_vec(x):
    """(B,1,28,28) -> (B,784)"""
    return x.view(x.size(0), -1)

def vec_to_image(x):
    """(784,) -> (28,28)  for a single example"""
    return x.view(28, 28)

class ReshapeToImage(nn.Module):
    """(B,784) -> (B,1,28,28)"""
    def forward(self, x):
        return x.view(-1, 1, 28, 28)


## 4. Architecture

### 4.1 — 2 Convolutional Blocks (`ARCH = '2blocks'`)

**Encoder**

Takes `x` (784,) and returns `(mu, logvar)` of dimension `LATENT_DIM`.
```
x (784) -> reshape (1,28,28)
         -> Conv2d(1->32, k=4, s=2, p=1)  -> (32,14,14)
         -> Conv2d(32->64, k=4, s=2, p=1) -> (64,7,7)
         -> Flatten -> (3136)
         -> Linear -> mu (LATENT_DIM)
         -> Linear -> logvar (LATENT_DIM or 1)
```

**Decoder**
Takes `z` (LATENT_DIM,) and returns `x_hat` (784,).
```
z (LATENT_DIM) -> Linear -> (3136) -> reshape (64,7,7)
               -> ConvTranspose2d(64->32, k=4, s=2, p=1) -> (32,14,14)
               -> ConvTranspose2d(32->1,  k=4, s=2, p=1) -> (1,28,28)
               -> Sigmoid -> Flatten -> x_hat (784)
```

In [ ]:
# ── 4.1  Encoder / Decoder — 2 convolutional blocks ──────────────────────────

class Encoder2(nn.Module):
    """
    VAE Encoder — 2 convolutional blocks.
    Input  : x of shape (B, 784)
    Output : mu (B, latent_dim) and logvar (B, latent_dim) or (B, 1)
    """
    def __init__(self, latent_dim, use_diff_sigma_E):
        super().__init__()
        self.conv = nn.Sequential(
            ReshapeToImage(),                                       # -> (B,1,28,28)
            nn.Conv2d(1,  32, kernel_size=4, stride=2, padding=1),  # -> (B,32,14,14)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # -> (B,64,7,7)
            nn.ReLU(),
            nn.Flatten(),                                           # -> (B,3136)
        )
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        # use_diff_sigma_E=True  -> different sigma per dimension (more expressive)
        # use_diff_sigma_E=False -> shared (scalar) sigma
        logvar_out = latent_dim if use_diff_sigma_E else 1
        self.fc_logvar = nn.Linear(64 * 7 * 7, logvar_out)

    def forward(self, x):
        h = self.conv(x)
        return self.fc_mu(h), self.fc_logvar(h)  # returns (mu, logvar)


class Decoder2(nn.Module):
    """
    VAE Decoder — 2 convolutional blocks.
    Input  : z of shape (B, latent_dim)
    Output : x_hat of shape (B, 784), values in [0,1]
    """
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7),
            nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),                                    # -> (B,64,7,7)
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # -> (B,32,14,14)
            nn.ReLU(),
            nn.ConvTranspose2d(32,  1, kernel_size=4, stride=2, padding=1),  # -> (B,1,28,28)
            nn.Sigmoid(),
            nn.Flatten(),                                                    # -> (B,784)
        )

    def forward(self, z):
        return self.net(z)

### 4.2 — 3 Convolutional Blocks (`ARCH = '3blocks'`)



**Encoder**

Takes `x` (784,) and returns `(mu, logvar)` of dimension `LATENT_DIM`.
```
x (784) -> reshape (1,28,28)
         -> Conv2d(1->32,   k=4, s=2, p=1)  -> (32,14,14)
         -> Conv2d(32->64,  k=4, s=2, p=1)  -> (64,7,7)
         -> Conv2d(64->128, k=4, s=2, p=1)  -> (128,3,3)
         -> Flatten -> (1152)
         -> Linear -> mu (LATENT_DIM)
         -> Linear -> logvar (LATENT_DIM or 1)
```

**Decoder**

Takes `z` (LATENT_DIM,) and returns `x_hat` (784,).
```
z (LATENT_DIM) -> Linear -> (1152) -> reshape (128,3,3)
               -> ConvTranspose2d(128->64, k=4, s=2, p=1, op=1) -> (64,7,7)
               -> ConvTranspose2d(64->32,  k=4, s=2, p=1)       -> (32,14,14)
               -> ConvTranspose2d(32->1,   k=4, s=2, p=1)       -> (1,28,28)
               -> Sigmoid -> Flatten -> x_hat (784)
```

In [ ]:
# ── 4.2  Encoder / Decoder — 3 convolutional blocks ──────────────────────────

class Encoder3(nn.Module):
    """VAE Encoder — 3 convolutional blocks (same convention as Encoder2)."""
    def __init__(self, latent_dim, use_diff_sigma_E):
        super().__init__()
        self.conv = nn.Sequential(
            ReshapeToImage(),                                       # -> (B,1,28,28)
            nn.Conv2d(1,   32, kernel_size=4, stride=2, padding=1), # -> (B,32,14,14)
            nn.ReLU(),
            nn.Conv2d(32,  64, kernel_size=4, stride=2, padding=1), # -> (B,64,7,7)
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # -> (B,128,3,3)
            nn.ReLU(),
            nn.Flatten(),                                           # -> (B,1152)
        )
        self.fc_mu = nn.Linear(128 * 3 * 3, latent_dim)
        logvar_out = latent_dim if use_diff_sigma_E else 1
        self.fc_logvar = nn.Linear(128 * 3 * 3, logvar_out)

    def forward(self, x):
        h = self.conv(x)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder3(nn.Module):
    """VAE Decoder — 3 convolutional blocks (mirrors Encoder3)."""
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128 * 3 * 3),
            nn.ReLU(),
            nn.Unflatten(1, (128, 3, 3)),                                  # -> (B,128,3,3)
            # 3 -> 7 needs output_padding=1 (7 is odd); the others don't.
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2,
                               padding=1, output_padding=1),               # -> (B,64,7,7)
            nn.ReLU(),
            nn.ConvTranspose2d(64,  32, kernel_size=4, stride=2, padding=1),# -> (B,32,14,14)
            nn.ReLU(),
            nn.ConvTranspose2d(32,   1, kernel_size=4, stride=2, padding=1),# -> (B,1,28,28)
            nn.Sigmoid(),
            nn.Flatten(),                                                  # -> (B,784)
        )

    def forward(self, z):
        return self.net(z)

In [ ]:
# ── Architecture factory ───────────────────────────────────────────────

_ARCHITECTURES = {
    '2blocks': (Encoder2, Decoder2),
    '3blocks': (Encoder3, Decoder3),
}

def get_encoder_decoder(arch, latent_dim, use_diff_sigma_E):
    """Return (encoder, decoder) instances for the requested architecture."""
    if arch not in _ARCHITECTURES:
        raise ValueError(f"Unknown architecture '{arch}'. Choose from {list(_ARCHITECTURES)}")
    EncoderCls, DecoderCls = _ARCHITECTURES[arch]
    return EncoderCls(latent_dim, use_diff_sigma_E), DecoderCls(latent_dim)

## 5. VAE (training wrapper)

In [33]:
class VAE(nn.Module):
    """
    Combines Encoder and Decoder.
    The forward method computes the ELBO loss:
        loss = reconstruction_error + KL_divergence
    """
    def __init__(self, latent_dim, use_diff_sigma_E, reconstruction_type, arch='2blocks'):
        super().__init__()
        self.latent_dim          = latent_dim
        self.use_diff_sigma_E    = use_diff_sigma_E
        self.reconstruction_type = reconstruction_type
        self.arch                = arch

        self.encoder, self.decoder = get_encoder_decoder(arch, latent_dim, use_diff_sigma_E)

    # ── Reparameterization trick ─────────────────────────────
    def sample_z(self, mu, logvar):
        """
        Sample z ~ N(mu, sigma^2) in a differentiable way.
        z = mu + sigma * epsilon,  epsilon ~ N(0,I)
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(mu)
        return mu + std * eps

    def forward(self, x):
        mu, logvar = self.encoder(x)
        var = logvar.exp()
        z   = self.sample_z(mu, logvar)
        x_hat = self.decoder(z)

        # — Reconstruction error (negative log-likelihood) —
        if self.reconstruction_type == 'Gaussian':
            # p(x|z) = N(x_hat, I)  ->  MSE
            recon = 0.5 * ((x_hat - x) ** 2).sum(dim=1).mean()

        elif self.reconstruction_type == 'Laplacian':
            # p(x|z) = Laplace(x_hat, 1)  ->  MAE
            recon = torch.abs(x_hat - x).sum(dim=1).mean()

        elif self.reconstruction_type == 'Bernoulli':
            # p(x|z) = Bernoulli(x_hat)  ->  BCE
            recon = -(
                (1e-4 + x_hat).log() * x +
                (1e-4 + 1 - x_hat).log() * (1 - x)
            ).sum(dim=1).mean()

        # — KL divergence: D_KL( q(z|x) || N(0,I) ) —
        if self.use_diff_sigma_E:
            # different sigma per dimension
            kl = 0.5 * (mu ** 2 + var - logvar - 1).sum(dim=1).mean()
        else:
            # shared scalar sigma
            kl  = 0.5 * (mu ** 2).sum(dim=1).mean()
            kl += 0.5 * self.latent_dim * (var - logvar - 1).mean()

        return recon + kl

    # ── Helper methods ───────────────────────────────────
    def encode(self, x):
        """Returns only mu (no noise) — used in the Diffusion stage."""
        mu, _ = self.encoder(x)
        return mu

    def decode(self, z):
        return self.decoder(z)


## 6. EMNIST dataset

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.EMNIST(
    root='data',
    split="byclass",   # 'byclass': 62 classes (0-9, A-Z, a-z)
    train=True,
    transform=transform,
    download=True,
)
test_dataset = datasets.EMNIST(
    root='data',
    split="byclass",
    train=False,
    transform=transform,
    download=True,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train  : {len(train_dataset):,} images')
print(f'Test   : {len(test_dataset):,} images')
print(f'Classes: {len(train_dataset.classes)}')

Train  : 697,932 images
Test   : 116,323 images
Classes: 62


In [ ]:
def fix_emnist_image(img: np.ndarray) -> np.ndarray:
    """Undo EMNIST's built-in 90° CW rotation + horizontal mirror (visualization only)."""
    return np.rot90(np.fliplr(img))

In [ ]:
images, labels = next(iter(train_loader))

titles = [f'{train_dataset.classes[labels[i].item()]} ({labels[i].item()})'
          for i in range(9)]
fig = make_subplots(rows=3, cols=3, subplot_titles=titles,
                    horizontal_spacing=0.03, vertical_spacing=0.10)
for i in range(9):
    r, c = i // 3 + 1, i % 3 + 1
    img = fix_emnist_image(images[i].squeeze(0).numpy())
    fig.add_trace(go.Heatmap(z=img, colorscale='gray', showscale=False),
                  row=r, col=c)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange='reversed', scaleanchor=None)
fig.update_layout(title_text='EMNIST train examples', height=680, width=680)
fig.show()

## 7. Training and evaluation functions

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, _ in loader:
        x = image_to_vec(images.to(device))
        optimizer.zero_grad()
        loss = model(x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    # loss is already the per-sample mean within each batch,
    # so average over the number of batches (not over the dataset size)
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    for images, _ in loader:
        x = image_to_vec(images.to(device))
        total_loss += model(x).item()
    return total_loss / len(loader)

## 8. Visualization during training

In [ ]:
@torch.no_grad()
def show_examples(model, fixed_images, epoch, device):
    """Show 3 originals, 3 reconstructions and 3 randomly generated samples."""
    clear_output(wait=True)
    model.eval()

    x     = image_to_vec(fixed_images[:3].to(device))
    x_hat = model.decode(model.encode(x))            # reconstruction
    z_rnd = torch.randn(3, model.latent_dim).to(device)
    x_gen = model.decode(z_rnd)                       # random generation

    x, x_hat, x_gen = x.cpu(), x_hat.cpu(), x_gen.cpu()

    rows   = [x, x_hat, x_gen]
    titles = ['Original', 'Reconstructed', 'Generated z~N(0,I)']

    # title only over the middle column of each row
    subplot_titles = []
    for title in titles:
        subplot_titles += ['', title, '']

    fig = make_subplots(rows=3, cols=3, subplot_titles=subplot_titles,
                        horizontal_spacing=0.02, vertical_spacing=0.08)
    for row_idx, row_data in enumerate(rows):
        for col_idx in range(3):
            img = fix_emnist_image(vec_to_image(row_data[col_idx]).numpy())
            fig.add_trace(go.Heatmap(z=img, colorscale='gray', showscale=False),
                          row=row_idx + 1, col=col_idx + 1)

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False, autorange='reversed')
    fig.update_layout(title_text=f'Epoch {epoch}/{EPOCHS}', height=720, width=720)
    fig.show()

## 9. Training loop

In [ ]:
def load_checkpoint(ckpt_name, device=device, optimizer=None):
    """Load a checkpoint and rebuild the VAE from the config stored inside it.

    Args:
        ckpt_name : checkpoint file name (e.g. 'vae_best.ckpt') or full path.
        device    : device to map the model to.
        optimizer : optional optimizer to restore the saved state into.

    Returns:
        model : VAE with encoder/decoder weights loaded, in eval mode.
        ckpt  : the raw checkpoint dict (has 'epoch', 'train_loss', 'test_loss', ...).
    """
    path = ckpt_name if os.path.isabs(ckpt_name) or os.path.sep in ckpt_name \
        else os.path.join(CKPT_DIR, ckpt_name)

    ckpt = torch.load(path, map_location=device)

    arch = ckpt.get('arch', '2blocks')   # backward compat with older checkpoints

    model = VAE(
        ckpt['latent_dim'],
        ckpt['use_diff_sigma_E'],
        ckpt['reconstruction'],
        arch,
    ).to(device)
    model.encoder.load_state_dict(ckpt['encoder'])
    model.decoder.load_state_dict(ckpt['decoder'])
    model.eval()

    if optimizer is not None:
        optimizer.load_state_dict(ckpt['optimizer'])

    print(f'Loaded {path}')
    print(f"  arch       : {arch}")
    print(f"  epoch      : {ckpt['epoch']}")
    print(f"  train_loss : {ckpt['train_loss']:.4f}")
    print(f"  test_loss  : {ckpt['test_loss']:.4f}")

    return model, ckpt


def init_training():
    """Build (or resume) the training state.

    If `vae_best.ckpt` exists, resume model, optimizer, epoch counter,
    best test loss and the loss-curve history from it. Otherwise start fresh.

    Returns:
        model, optimizer, start_epoch, best_test_loss, history
        where `history` is a dict with 'epoch_list', 'train_loss_list',
        'test_loss_list'.
    """
    best_ckpt_path = os.path.join(CKPT_DIR, 'vae_best.ckpt')

    if os.path.exists(best_ckpt_path) and GET_EARLY_TRAINED_MODEL:
        model, ckpt = load_checkpoint('vae_best.ckpt')
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        optimizer.load_state_dict(ckpt['optimizer'])
        start_epoch    = ckpt['epoch'] + 1
        best_test_loss = ckpt['test_loss']
        hist = ckpt.get('history') or {}
        history = {
            'epoch_list'      : hist.get('epoch_list', []),
            'train_loss_list' : hist.get('train_loss_list', []),
            'test_loss_list'  : hist.get('test_loss_list', []),
        }
        print(f'Resuming training from epoch {start_epoch} '
            f'(best test loss so far: {best_test_loss:.4f})')
    else:
        model = VAE(LATENT_DIM, USE_DIFF_SIGMA_E, RECONSTRUCTION, ARCH).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        start_epoch    = 1
        best_test_loss = float('inf')
        history = {'epoch_list': [], 'train_loss_list': [], 'test_loss_list': []}
        print(f'No checkpoint found \u2014 training from scratch (arch={ARCH}).')

    return model, optimizer, start_epoch, best_test_loss, history


# Example:
# model, ckpt = load_checkpoint('vae_best.ckpt')
# print('saved at epoch', ckpt['epoch'])


In [ ]:
# Handle Checkpointing ──────────────────────────────────────────────────
def save_checkpoint(path, model, optimizer, epoch, train_loss, test_loss, history=None):
    """Save encoder/decoder weights plus the config needed to rebuild them.

    `history` is an optional dict with the loss curves
    (epoch_list / train_loss_list / test_loss_list) so the plot stays
    continuous across resumes.
    """
    torch.save({
        'epoch'              : epoch,
        'encoder'            : model.encoder.state_dict(),
        'decoder'            : model.decoder.state_dict(),
        'optimizer'          : optimizer.state_dict(),
        'latent_dim'         : LATENT_DIM,
        'use_diff_sigma_E'   : USE_DIFF_SIGMA_E,
        'reconstruction'     : RECONSTRUCTION,
        'arch'               : ARCH,
        'train_loss'         : train_loss,
        'test_loss'          : test_loss,
        'history'            : history if history is not None else {},
    }, path)

def prune_old_checkpoints(keep_last):
    """Delete the oldest periodic checkpoints, keeping only the newest `keep_last`."""
    ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'vae_epoch_*.ckpt')))
    for old in ckpts[:-keep_last]:
        os.remove(old)
        print(f'  \u21b3 removed old checkpoint: {old}')
# ───────────────────────────────────────────────────────────


In [ ]:
class EarlyStopping:
    """Soft early stopping. `step(test_loss)` returns (should_stop, reason).

    Stops when EITHER:
      1) NO NEW MINIMUM: no new best test loss for `patience` epochs
         (catches overfitting), or
      2) PLATEAU: spread (max-min) over the last `patience` epochs is
         below `min_delta` (catches convergence).

    `initial_best` (e.g. the best loss restored from a checkpoint) keeps a
    resume from stopping early.
    """

    def __init__(self, patience, min_delta, enabled=True, initial_best=float('inf')):
        self.patience, self.min_delta, self.enabled = patience, min_delta, enabled
        self.best, self.since_best = initial_best, 0
        self.window = []   # last `patience` test losses

    def step(self, test_loss):
        if test_loss < self.best:
            self.best, self.since_best = test_loss, 0
        else:
            self.since_best += 1
        self.window.append(test_loss)
        self.window = self.window[-self.patience:]   # keep only the last `patience`

        if not self.enabled:
            return False, None
        if self.since_best >= self.patience:
            return True, f'no new minimum for {self.patience} epochs (best: {self.best:.4f})'
        if len(self.window) == self.patience:
            spread = max(self.window) - min(self.window)
            if spread < self.min_delta:
                return True, (f'loss flat — spread {spread:.4f} (< {self.min_delta}) '
                              f'over {self.patience} epochs')
        return False, None


In [ ]:

model, optimizer, start_epoch, best_test_loss, history = init_training()

epoch_list      = history['epoch_list']
train_loss_list = history['train_loss_list']
test_loss_list  = history['test_loss_list']

early_stopping = EarlyStopping(
    EARLY_STOPPING_PATIENCE, EARLY_STOPPING_MIN_DELTA,
    enabled=EARLY_STOPPING, initial_best=best_test_loss,
)

fixed_images, _ = next(iter(train_loader))
fixed_images = fixed_images[:3]

for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    test_loss  = evaluate(model, test_loader, device)

    epoch_list.append(epoch)
    train_loss_list.append(train_loss)
    test_loss_list.append(test_loss)

    print(f'Epoch [{epoch:03d}/{EPOCHS}]  '
          f'Train: {train_loss:.4f}  '
          f'Test: {test_loss:.4f}')

    # periodic checkpoint
    if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
        ckpt_path = os.path.join(CKPT_DIR, f'vae_epoch_{epoch:03d}.ckpt')
        save_checkpoint(ckpt_path, model, optimizer, epoch, train_loss, test_loss, history)
        prune_old_checkpoints(KEEP_LAST)
        print(f'  ↳ checkpoint saved: {ckpt_path}')

    # best model so far (survives a Colab disconnect)
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        save_checkpoint(os.path.join(CKPT_DIR, 'vae_best.ckpt'), model, optimizer, epoch, train_loss, test_loss, history)

    # early stopping
    stop, reason = early_stopping.step(test_loss)
    if stop:
        print(f'\nEarly stopping at epoch {epoch}: {reason}.')
        print(f'Best test loss: {best_test_loss:.4f} (saved as {CKPT_DIR}/vae_best.ckpt)')
        break

    if epoch % 5 == 0 or epoch == 1:
        show_examples(model, fixed_images, epoch, device)

print('Training complete!')
print(f'Best test loss: {best_test_loss:.4f}  (saved as {CKPT_DIR}/vae_best.ckpt)')


NameError: name 'GET_EARLY_TRAINED_MODEL' is not defined

## 10. Loss curve (for the presentation)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=epoch_list, y=train_loss_list,
                         mode='lines', name='Train loss'))
fig.add_trace(go.Scatter(x=epoch_list, y=test_loss_list,
                         mode='lines', name='Test loss',
                         line=dict(dash='dash')))
fig.update_layout(
    title=f'VAE EMNIST | {RECONSTRUCTION} | latent_dim={LATENT_DIM}',
    xaxis_title='Epoch',
    yaxis_title='Mean loss (negative ELBO)',
    width=900, height=400,
)
fig.update_xaxes(showgrid=True, gridcolor='rgba(0,0,0,0.1)')
fig.update_yaxes(showgrid=True, gridcolor='rgba(0,0,0,0.1)')

os.makedirs(RESULTS_DIR, exist_ok=True)
fig.write_image(os.path.join(RESULTS_DIR, 'vae_loss_curve.png'), scale=2)
fig.show()

## 11. 2D marginals of the latent space (for the presentation)

Projects all test set examples into the latent space
and plots scatter plots of pairs of dimensions colored by class.
This shows whether the VAE learned a good class-separated representation.

In [27]:
@torch.no_grad()
def plot_latent_marginals(model, loader, device, n_pairs=None, max_samples=5000):
    """
    Plots scatter plots of pairs of latent space dimensions.
    Each point is colored by the image class.
    """
    model.eval()
    all_z, all_labels = [], []

    for images, labels in loader:
        x = image_to_vec(images.to(device))
        z = model.encode(x)   # uses only mu, no noise
        all_z.append(z.cpu())
        all_labels.append(labels)
        if sum(len(b) for b in all_z) >= max_samples:
            break

    Z      = torch.cat(all_z)[:max_samples].numpy()
    labels = torch.cat(all_labels)[:max_samples].numpy()

    # derive n_pairs from the actual latent dimension
    latent_dim = Z.shape[1]
    if n_pairs is None:
        n_pairs = latent_dim // 2
    n_pairs = min(n_pairs, latent_dim // 2)  # safety clamp

    # plot n_pairs pairs of dimensions: (0,1), (2,3), ...
    titles = [f'Dims {2 * i} vs {2 * i + 1}' for i in range(n_pairs)]
    # equal width per subplot so each panel is rendered as a square
    panel_px = 420
    fig = make_subplots(rows=1, cols=n_pairs, subplot_titles=titles,
                        horizontal_spacing=0.06)

    for i in range(n_pairs):
        d1, d2 = 2 * i, 2 * i + 1
        # per-pair symmetric range: same for x and y so axes are equal
        pair_max = float(max(np.abs(Z[:, d1]).max(), np.abs(Z[:, d2]).max()))
        r = pair_max * 1.05
        ax_range = [-r, r]
        fig.add_trace(
            go.Scattergl(
                x=Z[:, d1], y=Z[:, d2], mode='markers',
                marker=dict(size=3, color=labels, colorscale='Turbo',
                            opacity=0.5, cmin=0, cmax=61,
                            showscale=(i == n_pairs - 1),
                            colorbar=dict(title='class')),
                showlegend=False,
            ),
            row=1, col=i + 1,
        )
        fig.update_xaxes(title_text=f'z[{d1}]', range=ax_range,
                         constrain='domain', row=1, col=i + 1)
        fig.update_yaxes(title_text=f'z[{d2}]', range=ax_range,
                         constrain='domain',
                         scaleanchor=f'x{i + 1}' if i > 0 else 'x',
                         scaleratio=1, row=1, col=i + 1)

    fig.update_layout(
        title_text='2D marginals of the latent space z — test set',
        width=panel_px * n_pairs + 80, height=panel_px,
    )
    os.makedirs(RESULTS_DIR, exist_ok=True)
    fig.write_image(os.path.join(RESULTS_DIR, 'latent_marginals.png'), scale=2)
    fig.show()

# Prefer the best checkpoint; if it is missing, fall back to the in-memory model
if os.path.exists(os.path.join(CKPT_DIR, 'vae_best.ckpt')):
    plot_model, _ = load_checkpoint('vae_best.ckpt')
elif 'model' in globals():
    print('vae_best.ckpt not found — using the in-memory model from the last run.')
    plot_model = model
else:
    raise FileNotFoundError(
        'No vae_best.ckpt and no model in memory. '
        'Run the training cell first.'
    )

plot_latent_marginals(plot_model, test_loader, device)

Loaded training_checkpoints/vae_best.ckpt
  epoch      : 95
  train_loss : 179.6544
  test_loss  : 179.9375
